# LUAD full-gene decoder training

Train one count-aware decoder from the canonical scanVI AnnData object. The same checkpoint supplies ligand-receptor expression during Stage 2 and full-gene expected counts for downstream analysis.


## Contract

- Input: `artifacts/checkpoints/scanvi/adata.h5ad`
- Latent representation: `adata.obsm[latent_key]`
- Target expression: integer raw counts in `adata.layers['counts']`
- Output pattern: `artifacts/checkpoints/decoder/checkpoints/{src}_{tgt}.pt`
- Each route trains a full-gene decoder using only its source and target samples.
- Time is **not** an input to the decoder. The numeric values in `sample_times` only identify sample groups for reproducible train, validation, and test splits.


In [ ]:
from pathlib import Path
import sys

DATASET = "LUAD"
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
sys.path.append(str(REPO_ROOT / "src"))

import anndata as ad
import numpy as np
import yaml

from stvirtual.decoder.config import TrainConfig
from stvirtual.decoder.train import train_decoder


## Resolve paths and training settings


In [ ]:
with (EXPERIMENT_DIR / "config.yaml").open(encoding="utf-8") as handle:
    experiment_config = yaml.safe_load(handle)

def resolve_experiment_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

def decoder_checkpoint(src, tgt):
    pattern = experiment_config["decoder_checkpoint"]
    return resolve_experiment_path(pattern.format(src=src, tgt=tgt))

ROUTES = [('AAH', 'LUAD')]
DATA_PATH = resolve_experiment_path(experiment_config["scanvi_dir"]) / "adata.h5ad"
DECODER_OUTPUT_DIR = decoder_checkpoint(*ROUTES[0]).parents[1]
LATENT_KEY = experiment_config.get("latent_key", "X_scanVI")
COUNTS_KEY = "counts"
SAMPLE_KEY = "status"
DEVICE = "cuda:0"
SEED = 2025

print(f"Input AnnData: {DATA_PATH}")
print("Decoder routes:", ROUTES)
for src, tgt in ROUTES:
    print(f"  {src} -> {tgt}: {decoder_checkpoint(src, tgt)}")


## Audit the canonical AnnData object


In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run preprocess.ipynb first to create the canonical AnnData object."
    )

adata = ad.read_h5ad(DATA_PATH, backed="r")
try:
    if LATENT_KEY not in adata.obsm:
        raise KeyError(f"adata.obsm does not contain {LATENT_KEY!r}")
    if COUNTS_KEY not in adata.layers:
        raise KeyError(f"adata.layers does not contain {COUNTS_KEY!r}")
    if SAMPLE_KEY not in adata.obs:
        raise KeyError(f"adata.obs does not contain {SAMPLE_KEY!r}")
    sample_names = list(dict.fromkeys(adata.obs[SAMPLE_KEY].astype(str)))
    missing_samples = sorted({sample for route in ROUTES for sample in route} - set(sample_names))
    if missing_samples:
        raise ValueError(f"Decoder routes reference missing samples: {missing_samples}")
    summary = {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "latent_shape": tuple(adata.obsm[LATENT_KEY].shape),
        "available_samples": sample_names,
        "routes": ROUTES,
    }
finally:
    adata.file.close()

summary


## Configure training

The decoder predicts negative-binomial expected counts for every gene. Adjust the optimization settings here if the dataset requires a smaller batch or an earlier stopping window.


In [ ]:
TRAINING_KWARGS = dict(
    data_path=DATA_PATH,
    output_dir=DECODER_OUTPUT_DIR,
    latent_key=LATENT_KEY,
    counts_key=COUNTS_KEY,
    sample_key=SAMPLE_KEY,
    rollout_normalization_checkpoint=None,
    device=DEVICE,
    epochs=100,
    patience=20,
    batch_size=256,
    learning_rate=3e-4,
    seed=SEED,
    overwrite=False,
)
TRAINING_KWARGS


## Train the decoder


In [ ]:
checkpoints = {}
for src, tgt in ROUTES:
    decoder_config = TrainConfig(
        **TRAINING_KWARGS,
        sample_times={src: 0.0, tgt: 1.0},
        checkpoint_name=f"{src}_{tgt}.pt",
    )
    checkpoint = train_decoder(decoder_config).resolve()
    expected = decoder_checkpoint(src, tgt).resolve()
    if checkpoint != expected:
        raise RuntimeError(f"Decoder output {checkpoint} does not match config path {expected}")
    checkpoints[f"{src}_to_{tgt}"] = checkpoint

checkpoints


## Verify checkpoint metadata


In [ ]:
import torch

audits = {}
for route, checkpoint in checkpoints.items():
    payload = torch.load(checkpoint, map_location="cpu", weights_only=False)
    audit = {
        "checkpoint": str(checkpoint),
        "time_conditioning": payload.get("time_conditioning"),
        "latent_key": payload.get("latent_key"),
        "counts_key": payload.get("counts_key"),
        "n_genes": len(payload.get("gene_names", [])),
        "best_validation_loss": payload.get("best_validation_loss"),
    }
    if audit["time_conditioning"] is not False:
        raise ValueError("The public decoder must be trained without time conditioning")
    audits[route] = audit

audits


## Outputs

Each route produces `{src}_{tgt}.pt` under `artifacts/checkpoints/decoder/checkpoints/`. Its route-specific configuration, training history, and scale audit are stored beside the checkpoint directory without overwriting other routes. Stage 2 resolves the same path pattern from this dataset's `config.yaml`.
